# Financial Product Portfolio Construction

In [2]:
import src.utils.file_management as filemgmt
from src.pipeline.preprocessing import StockPriceDataManager
from src.pipeline.financial_products import KOCertificate, KOCertificateSet

import pandas as pd
import numpy as np

from pathlib import Path
from tqdm import tqdm

In [3]:
ROOT = Path().resolve().parent

DATA = ROOT / "data"
DOWNLOADED_PRICES = DATA / "minutely_price_downloads"
INTERPOLATED_PRICES = DATA / "interpolated_prices_dax"

SAVED_PORTFOLIOS = DATA / "portfolios"

PRIVATE_FILES = ROOT / "private"
AV_API_KEY_FILE = PRIVATE_FILES / "Alpha Vantage API Key.txt"
with open(AV_API_KEY_FILE) as file: AV_API_KEY = file.read()

## DataManager

In [4]:
data_manager = StockPriceDataManager(ticker_symbol='DAX',
                                    download_dir=DOWNLOADED_PRICES,
                                    interpolated_files_dir=INTERPOLATED_PRICES,
                                    alpha_vantage_api_key=AV_API_KEY,
                                    env_sampling_rate_minutes=15,
                                    is_etf_price_data=True,
                                    non_etf_time_price_tuples=[('2025-06-04 18:00:00', 24276.48),
                                                               ('2025-06-02 17:00:00', 23942.52),
                                                               ('2025-06-03 12:00:00', 23962.40),
                                                               ('2025-05-30 17:00:00', 23997.71)]
                                    )

## Scrape Real Products
Wikifolio trades ko-certificates from Societe Generale and HSBC.

In [ ]:
ko_certificate_list = []
for isin, direction, date_price in tqdm(zip(
        ["DE000TT2CH77", "DE000HS3U0K3", "DE000TB192G9", "DE000TT0ZP72",
         "DE000TT1WHV8", "DE000HG75555", "DE000HS2W424", "DE000HS3GYD5", "DE000HS8DSJ0",
         "DE000HT42LT9", "DE000FA61D63", "DE000FA0RD26", "DE000FA0Q232", "DE000SX0LHS4",
         "DE000SX0LHS4", "DE000HT52AV7", "DE000HT52AV7"],
        ["long", "long", "long", "long",
         "long", "long", "long", "long", "long",
         "long", "short", "short", "short", "short",
         "short", "short", "short"],
        [("2022-09-26", 6.49), ("2024-08-05", 0.81), ("2022-09-26", 85.27), ("2023-10-23", 62.97),
         ("2023-10-23", 36.33), ("2023-10-23", 15.98), ("2024-08-23", 2.22), ("2024-08-05", 1.34), ("2025-03-31", 2.38),
         ("2025-04-08", 0.83), ("2025-07-23", 111.01), ("2025-07-09", 93.60), ("2025-07-09", 78.45),
         ("2025-03-18", 80.9),
         ("2025-03-18", 70.19), ("2025-07-09", 50.85), ("2025-07-09", 4.43)
         ])):
    try:
        ko_certificate = KOCertificate(
            underlying_price_series=data_manager.non_etf_env_interp_prices,
            isin=isin,
            direction=direction,
            historic_date_future_price_tuple=date_price
        )
        ko_certificate.fix_initial_knockout()
        ko_certificate_list.append(ko_certificate)
    except (KeyError, TypeError) as e:
        print(f"Error occurred for ISIN {isin}: {e}. Skipping this ISIN.")

portfolio = KOCertificateSet(ko_certificates=ko_certificate_list)

for product in portfolio.ko_certificates:
    product.fix_initial_knockout()

portfolio.save_to_csv(SAVED_PORTFOLIOS / filemgmt.file_title("Scraped Certificate Set", ".csv"))

## Instantiate Artificial Products

In [ ]:
# initialise for 2020/07, 2022/07, 2024/01
portfolio = KOCertificateSet(
    underlying_price_series=data_manager.non_etf_env_interp_prices,
    base_price_inference_timestamps=["2020-07-06 10:00:00", "2022-07-05 10:00:00", "2024-01-05 10:00:00"],
    # "2023-07-05 10:00:00", "2024-07-05 10:00:00", "2025-03-05 10:00:00"],
    n_products_per_direction=15,
    lowest_leverage=1.0, highest_leverage=10.0)

# add several other with more recent issue dates
portfolio.ko_certificates += portfolio.initialise_products_from_leverage(
    underlying_price_series=data_manager.non_etf_env_interp_prices,
    base_price_inference_timestamps=["2017-12-08 10:00:00", "2018-07-05 10:00:00"],
    n_products_per_direction=15, issue_date="2017-01-01 10:00:00",
    lowest_leverage=1.0, highest_leverage=10.0)

portfolio.ko_certificates += portfolio.initialise_products_from_leverage(
    underlying_price_series=data_manager.non_etf_env_interp_prices,
    base_price_inference_timestamps=["2021-07-06 10:00:00", "2023-07-05 10:00:00", "2024-07-05 10:00:00",
                                     "2025-03-05 10:00:00"],
    n_products_per_direction=15, issue_date="2020-06-01 10:00:00",
    lowest_leverage=1.0, highest_leverage=10.0)

portfolio.ko_certificates += portfolio.initialise_products_from_leverage(
    underlying_price_series=data_manager.non_etf_env_interp_prices,
    base_price_inference_timestamps=["2023-05-05 10:00:00", "2024-07-05 10:00:00", "2025-03-05 10:00:00"],
    n_products_per_direction=15, issue_date="2022-12-01 10:00:00",
    lowest_leverage=1.0, highest_leverage=10.0)

portfolio.ko_certificates += portfolio.initialise_products_from_leverage(
    underlying_price_series=data_manager.non_etf_env_interp_prices,
    base_price_inference_timestamps=["2025-03-05 12:00:00"],
    n_products_per_direction=15, issue_date="2024-03-03 10:00:00",
    lowest_leverage=1.0, highest_leverage=10.0)

portfolio.save_to_csv(SAVED_PORTFOLIOS / filemgmt.file_title("Artificial Certificate Set", ".csv"))